# Laboratório 05: Dados Ausentes - Tratamento Univariado e Multivariado
**Disciplina:** Extração e Preparação de Dados (IBM8915)
**Objetivo:** Aprender a usar ferramentas de descarte (`dropna`), preenchimento simples (`fillna`) com estatísticas básicas (univariado) e algoritmos de machine learning via Scikit-Learn (`KNNImputer`) (multivariado) para tratamento assertivo de dados defeituosos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

## Parte 1: Exemplo Guiado - A "Faca" e a "Seringa"
Neste exemplo rápido, usaremos o `.dropna()` (a faca) para amputar o que não tem salvação e o `.fillna()` (a seringa) para curar colunas numéricas e categóricas do nosso DataFrame de pacientes.

In [ ]:
# Criando dados fictícios
df_exemplo = pd.DataFrame({
    'Paciente': ['Ana', 'Bruno', 'Carlos', 'Diana', 'Eduardo'],
    'Idade': [25, np.nan, 42, 38, np.nan],         # Numérica
    'Pressao': [120, 125, 130, np.nan, 140],       # Numérica
    'Tipo_Sanguineo': ['O+', 'A-', np.nan, 'O+', np.nan], # Categórica
    'Coluna_Perdida': [np.nan, np.nan, np.nan, np.nan, np.nan] # 100% Nula
})

print("Estado Original:\n", df_exemplo, "\n")

# 1. A Faca: Descartando colunas 100% vazias (axis=1 atinge colunas)
df_exemplo.dropna(axis=1, how='all', inplace=True)

# 2. A Seringa (Numéricos): Preenchendo Idade com a Mediana
mediana_idade = df_exemplo['Idade'].median()
df_exemplo['Idade'].fillna(mediana_idade, inplace=True)

# 3. A Seringa (Categóricos): Preenchendo Tipo Sanguíneo com a Moda
moda_sangue = df_exemplo['Tipo_Sanguineo'].mode()[0] #  pega a primeira moda se houver empate
df_exemplo['Tipo_Sanguineo'].fillna(moda_sangue, inplace=True)

print("Estado Tratado:\n", df_exemplo)

## Parte 2: Exercício Prático (Mão na Massa)
Abaixo geramos um pequeno dataset de imóveis (`df_imoveis`). 
**Sua Tarefa:**
1. Descarte as colunas que possuam **mais de 70%** de valores nulos.
2. Preencha a coluna numérica `Preco` usando a estatística adequada (Média ou Mediana? Dica: Verifique se existem outliers absurdos com `.describe()`).
3. Preencha a coluna categórica `Bairro` com o valor mais comum (Moda).

In [ ]:
# NÃO ALTERE ESTE CÓDIGO
np.random.seed(42)
df_imoveis = pd.DataFrame({
    'ID_Imovel': range(1, 101),
    'Bairro': np.random.choice(['Centro', 'Jardins', 'Vila Nova'], 100, p=[0.5, 0.3, 0.2]),
    'Preco': np.append(np.random.normal(300000, 50000, 95), [1500000, 2000000, 2500000, 3000000, 5000000]), # Com outliers milionários
    'Comissao_Corretor_Morta': [np.nan] * 100 # Coluna inútil
})

# Injetando nulos
df_imoveis.loc[np.random.choice(100, 15, replace=False), 'Preco'] = np.nan
df_imoveis.loc[np.random.choice(100, 20, replace=False), 'Bairro'] = np.nan

In [ ]:
# ESCREVA SEU CÓDIGO AQUI
# Passo 1: Use dropna para derrubar a 'Comissao_Corretor_Morta'


# Passo 2: Decida entre .mean() e .median() para 'Preco' e faça o fillna


# Passo 3: Use .mode() para preencher os bairros ausentes



---
## Parte 3: Tratamento Multivariado (Avançado com Scikit-Learn)
Em vez de olhar apenas para a coluna com valores ausentes (univariado), a imputação multivariada utiliza as **outras variáveis** da mesma linha para prever o valor faltante. Vamos usar o `KNNImputer`, que acha os exemplos mais "parecidos" (vizinhos) para inferir a lacuna.

In [ ]:
from sklearn.impute import KNNImputer

# Criando um pequeno DataFrame de exemplo
df_knn = pd.DataFrame({
    'Salario': [5000, 5200, 9000, 5100, 8900],
    'Idade': [25, 27, 45, 26, np.nan],  # Alguém esqueceu a idade do último funcionário
    'Tempo_Empresa': [1, 2, 15, 1, 14]
})
print("Antes da Imputação (KNN):\n", df_knn, "\n")

# O KNNImputer substitui o valor ausente usando a média dos k vizinhos mais próximos
imputador_knn = KNNImputer(n_neighbors=2)
df_imputado_numpy = imputador_knn.fit_transform(df_knn)

# O Scikit-Learn retorna um array NumPy! Precisamos recriar o DataFrame
df_knn_tratado = pd.DataFrame(df_imputado_numpy, columns=df_knn.columns)
print("Após a Imputação (KNN):\n", df_knn_tratado)

### Exercício Prático: KNNImputer
Lembra da coluna `Preco` do nosso dataset `df_imoveis` (na Parte 2)? Se apenas preenchermos com a mediana global, ignoramos se a casa é uma mansão gigante ou um estúdio diminuto.
Vamos preencher o Preco agora de forma inteligente, usando a correlação com as variáveis numéricas fictícias adicionadas: `Metragem` e `Numero_Quartos`.

In [ ]:
from sklearn.impute import KNNImputer

# 1. Instanciar
imputer = KNNImputer(n_neighbors=5)

# 2. Fit/Transform
df_imoveis_knn_imputed = imputer.fit_transform(df_imoveis_knn)

# 3. Recriar DataFrame
df_imoveis_knn_tratado = pd.DataFrame(df_imoveis_knn_imputed, columns=df_imoveis_knn.columns)

# 4. Exibir
print('Amostra após KNNImputer:')
display(df_imoveis_knn_tratado.head(10))


In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# Passo 1: Instancie o KNNImputer (use n_neighbors=3 ou 5 para teste)


# Passo 2: Aplique o fit_transform no df_imoveis_knn e guarde o resultado (que será um array numpy)


# Passo 3: Recrie o Pandas DataFrame com os nomes originais das colunas
# Dica: pd.DataFrame(sua_variavel_numpy, columns=df_imoveis_knn.columns)


# Passo 4: Exiba o novo dataset tratado para validar se os preços foram preenchidos (use .head(10))



---
## Parte 4: Desafio para Casa - O Pipeline do Engenheiro de Dados
Este desafio consolida as **Aulas 02 a 07**. Rode a célula abaixo para gerar um arquivo CSV no seu disco chamado `ecommerce_messy.csv`.

**A Missão:**
1. **(Aula 02):** Carregue o arquivo lidando com o separador `;` e o encoding `latin-1`.
2. **(Aula 05):** Audite e remova as linhas inteiramente duplicadas.
3. **(Aula 04):** Converta a coluna `Categoria` para o tipo `category` para salvar memória.
4. **(Aula 06):** Plote um `sns.heatmap` para ver onde estão os nulos.
5. **(Aula 07):** Faça a imputação: `Idade` (Numérica) e `Categoria` (Texto).
6. **(Aula 04):** Calcule o Ticket Médio (`Renda`) agrupado por `Categoria`.

In [ ]:
# 1. Carregamento
df_messy = pd.read_csv('ecommerce_messy.csv', sep=';', encoding='latin-1')

# 2. Duplicatas
df_messy.drop_duplicates(inplace=True)

# 3. Categoria
df_messy['Categoria'] = df_messy['Categoria'].astype('category')

# 4. Heatmap
plt.figure(figsize=(10,4))
sns.heatmap(df_messy.isnull(), cbar=False, cmap='viridis')
plt.title('Mapa de Calor de Nulos - E-commerce')
plt.show()

# 5. Imputação Univariada
df_messy['Idade'].fillna(df_messy['Idade'].median(), inplace=True)
df_messy['Categoria'].fillna(df_messy['Categoria'].mode()[0], inplace=True)

# 6. Agrupamento
print('\nTicket Médio (Renda) por Categoria:')
print(df_messy.groupby('Categoria')['Renda'].mean())


In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# 1. Carregue o CSV (read_csv com sep e encoding corretos)

# 2. Remova duplicatas (.drop_duplicates)

# 3. Otimização de tipo (.astype('category'))

# 4. Heatmap do Seaborn (sns.heatmap)

# 5. Tratamento Univariado (fillna com mediana/moda)

# 6. Análise Agrupada (groupby)


---
## Parte 5: Desafio Final - Sobreviventes do Titanic (KNNImputer)
Para fixar o conhecimento de imputação multivariada vamos usar o famoso dataset do Titanic, embutido na biblioteca Seaborn.
O problema crônico desse dataset é a coluna `age` (idade), que possui dezenas de valores nulos. Usar apenas a média geral distorce a realidade: passageiros da 1ª classe tendiam a ser mais velhos que os da 3ª classe, e pessoas que viajavam sozinhas tinham idades diferentes das que viajavam em grandes famílias.

**Sua Missão:**
1. Isolar um sub-dataset apenas com características numéricas relevantes: `survived`, `pclass`, `age`, `sibsp` (Irmãos/Cônjuges), `parch` (Pais/Filhos) e `fare` (Tarifa).
2. Instanciar o `KNNImputer` e treinar neste sub-dataset para preencher as idades.
3. Recriar o DataFrame e confirmar que não existem mais nulos na variável `age` usando `.info()` ou `.isna().sum()`.

In [ ]:
from sklearn.impute import KNNImputer

# 1. Isolar numéricos relevantes
colunas_alvo = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
df_titanic_num = df_titanic[colunas_alvo]

# 2. Instanciar
imputer = KNNImputer(n_neighbors=5)

# 3. Fit/Transform
df_titanic_imputed_numpy = imputer.fit_transform(df_titanic_num)

# 4. Recriar DataFrame
df_titanic_tratado = pd.DataFrame(df_titanic_imputed_numpy, columns=colunas_alvo)

# 5. Validar
print('Nulos após KNNImputer no Titanic:')
print(df_titanic_tratado.isna().sum())


In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# Passo 1: Crie df_titanic_num selecionando apenas ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']


# Passo 2: Instancie o KNNImputer (ex: 5 vizinhos)


# Passo 3: Aplique o imputer.fit_transform() no df_titanic_num


# Passo 4: Converta a matriz numpy de volta para um DataFrame do Pandas (lembre de passar columns=df_titanic_num.columns)


# Passo 5: Exiba a quantidade de nulos na nova base (.isna().sum()) para validar o sucesso!

